In [ ]:
"""
PrideConnect Live Recommender: Enterprise-Grade Training, Calibration,
Benchmarking, Drift Monitoring, and Candidate Publishing Pipeline.

Supports:
- Acceptance, Relevance, and Neural Collaborative Filtering (NCF) models
- Exponential time-decay sample weighting & ElasticNet / L1 / L2 regularization
- Expanding-window temporal cross-validation with bootstrap confidence intervals
- Post-hoc calibration diagnostics (Platt scaling, Isotonic, ECE, MCE, reliability bins)
- Multi-model benchmarking (HistGBDT, RandomForest, ExtraTrees, MLP, Prior)
- Ranking metrics (HitRate@K, NDCG@K, MRR, MAP@K, Lift@K)
- Covariate, prior, and prediction drift monitoring (PSI, KS, Wasserstein, JS-div)
- Champion-Challenger paired bootstrap significance testing
- Shadow/Candidate publishing after dry-run so the app can choose what to activate
"""

from __future__ import annotations

import argparse
import functools
import json
import logging
import math
import os
import random
import sys
import time
import unittest
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import requests
import torch
import torch.nn as nn
from requests.adapters import HTTPAdapter
from scipy.special import expit
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.neural_network import MLPClassifier
from urllib3.util.retry import Retry

try:
    from google.colab import drive, userdata  # type: ignore

    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# =============================================================================
# 1. CONFIGURATION & LOGGING
# =============================================================================


@dataclass
class PipelineConfig:
    """Centralized, validated configuration for the training pipeline."""

    seed: int = 17
    publish: bool = True
    dry_run: bool = False
    publish_after_dry_run: bool = True
    run_tests: bool = True
    run_id: str = field(
        default_factory=lambda: datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    )

    # Connectivity & Storage
    app_url: str = ""
    sync_token: str = ""
    drive_dir: Path = Path("/content/drive/MyDrive/KQ-PrideConnect/models")
    fallback_dir: Path = Path("/content/KQ-PrideConnect/models")
    http_timeout_connect: int = 10
    http_timeout_read: int = 120
    http_max_retries: int = 4

    # Linear / Tabular Modeling
    lr_c_grid: Tuple[float, ...] = (0.005, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0)
    lr_penalties: Tuple[str, ...] = ("l2", "l1", "elasticnet")
    lr_l1_ratios: Tuple[float, ...] = (0.25, 0.5, 0.75)
    lr_min_auc: float = 0.50
    lr_min_brier_skill: float = -0.05
    time_decay_half_life_days: Optional[float] = 90.0
    bootstrap_rounds: int = 500
    ece_bins: int = 10

    # NCF Modeling
    availability_features: Tuple[str, ...] = (
        "availabilityDayOverlap",
        "availabilityTimeOverlap",
        "availabilityFrequencyFit",
        "availabilityPairKnown",
    )
    ncf_embedding_dim: int = 8
    ncf_hidden_dim: int = 16
    ncf_max_rows: int = 6000
    ncf_min_rows: int = 80
    ncf_min_members: int = 12
    ncf_min_holdout: int = 20
    ncf_min_coverage: float = 0.40
    ncf_min_auc: float = 0.52
    ncf_max_epochs: int = 300
    ncf_batch_size: int = 256
    ncf_patience: int = 25
    ncf_lr_grid: Tuple[float, ...] = (0.003, 0.005, 0.01)
    ncf_weight_decays: Tuple[float, ...] = (1e-3, 1e-2, 5e-2)
    ncf_dropouts: Tuple[float, ...] = (0.0, 0.15)
    ncf_label_smoothing: float = 0.02
    ncf_grad_clip: float = 5.0

    # Drift & Champion-Challenger
    psi_warning_threshold: float = 0.10
    psi_critical_threshold: float = 0.25
    non_regression_tolerance: float = 0.02

    def validate(self) -> None:
        if not self.app_url.startswith("https://"):
            raise ValueError("app_url must start with 'https://'")
        if not self.sync_token:
            raise ValueError("sync_token must be provided")
        if self.ncf_embedding_dim <= 0 or self.ncf_hidden_dim <= 0:
            raise ValueError("NCF dimensions must be positive")
        if not (0.0 <= self.ncf_label_smoothing < 0.5):
            raise ValueError("ncf_label_smoothing must be in [0.0, 0.5)")


def setup_logging(log_dir: Optional[Path] = None, level: int = logging.INFO) -> logging.Logger:
    """Configure structured console and optional file logging."""
    logger = logging.getLogger("prideconnect.ml")
    logger.setLevel(level)
    logger.handlers.clear()

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%dT%H:%M:%SZ",
    )

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    if log_dir is not None:
        log_dir.mkdir(parents=True, exist_ok=True)
        file_handler = logging.FileHandler(log_dir / "pipeline.log", encoding="utf-8")
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

    return logger


def timed(stage_name: str) -> Callable:
    """Decorator to log execution duration of pipeline stages."""

    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            logger = logging.getLogger("prideconnect.ml")
            t0 = time.perf_counter()
            logger.info("Starting stage: %s", stage_name)
            try:
                result = func(*args, **kwargs)
                elapsed = time.perf_counter() - t0
                logger.info("Completed stage: %s (%.2fs)", stage_name, elapsed)
                return result
            except Exception:
                elapsed = time.perf_counter() - t0
                logger.exception("Failed stage: %s (%.2fs)", stage_name, elapsed)
                raise

        return wrapper

    return decorator


def seed_everything(seed: int) -> None:
    """Ensure deterministic behavior across Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def read_secret(name: str, prompt: str, hidden: bool = False) -> str:
    """Read secret from Colab userdata, environment variables, or interactive prompt."""
    value: Optional[str] = None
    if IN_COLAB:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = os.environ.get(name)
    if not value:
        value = getpass(prompt) if hidden else input(prompt)
    return (value or "").strip()


# =============================================================================
# 2. DATA INGESTION, SANITIZATION, WEIGHTING & DRIFT MONITORING
# =============================================================================


def to_float(value: Any, default: float = 0.0) -> float:
    """Safely convert arbitrary input to a finite float."""
    try:
        number = float(value)
    except (TypeError, ValueError):
        return default
    return number if math.isfinite(number) else default


def parse_timestamp(ts: Any) -> Optional[datetime]:
    """Parse ISO-8601 timestamp string into a UTC-aware datetime."""
    if not ts or not isinstance(ts, str):
        return None
    try:
        cleaned = ts.replace("Z", "+00:00")
        dt = datetime.fromisoformat(cleaned)
        return dt if dt.tzinfo else dt.replace(tzinfo=timezone.utc)
    except ValueError:
        return None


def usable_rows(rows: Optional[List[Dict[str, Any]]], needs_features: bool) -> List[Dict[str, Any]]:
    """Filter valid binary-labeled rows and sort chronologically."""
    kept: List[Dict[str, Any]] = []
    for row in rows or []:
        if row.get("label") not in (0, 1):
            continue
        if needs_features and not isinstance(row.get("features"), dict):
            continue
        kept.append(row)
    return sorted(kept, key=lambda r: r.get("timestamp") or "")


def design_matrix(rows: Sequence[Dict[str, Any]], features: Sequence[str]) -> np.ndarray:
    """Construct dense float64 design matrix from row feature dicts."""
    return np.array(
        [[to_float(row["features"].get(name)) for name in features] for row in rows],
        dtype=np.float64,
    )


def compute_time_decay_weights(
    rows: Sequence[Dict[str, Any]], half_life_days: Optional[float]
) -> np.ndarray:
    """Compute exponential recency weights normalized to mean 1.0."""
    n = len(rows)
    if n == 0 or half_life_days is None or half_life_days <= 0:
        return np.ones(n, dtype=np.float64)

    parsed = [parse_timestamp(r.get("timestamp")) for r in rows]
    valid_dts = [dt for dt in parsed if dt is not None]
    if not valid_dts:
        return np.ones(n, dtype=np.float64)

    max_dt = max(valid_dts)
    lam = math.log(2.0) / half_life_days
    weights = np.array(
        [
            math.exp(-lam * max(0.0, (max_dt - dt).total_seconds() / 86400.0))
            if dt is not None
            else 1.0
            for dt in parsed
        ],
        dtype=np.float64,
    )
    mean_w = float(np.mean(weights))
    return weights / mean_w if mean_w > 0 else np.ones(n, dtype=np.float64)


def compute_psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    """Compute Population Stability Index (PSI) between two 1D distributions."""
    if len(expected) < 5 or len(actual) < 5:
        return 0.0
    quantiles = np.linspace(0, 100, bins + 1)
    breakpoints = np.unique(np.percentile(expected, quantiles))
    if len(breakpoints) < 2:
        return 0.0
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    exp_counts = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    act_counts = np.histogram(actual, bins=breakpoints)[0] / len(actual)

    eps = 1e-6
    exp_pct = np.clip(exp_counts, eps, None)
    act_pct = np.clip(act_counts, eps, None)
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))


def compute_js_divergence(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    """Compute Jensen-Shannon divergence between two 1D distributions."""
    if len(expected) < 5 or len(actual) < 5:
        return 0.0
    lo = min(float(np.min(expected)), float(np.min(actual)))
    hi = max(float(np.max(expected)), float(np.max(actual)))
    if hi - lo < 1e-12:
        return 0.0
    edges = np.linspace(lo, hi, bins + 1)
    p = np.histogram(expected, bins=edges)[0] / len(expected)
    q = np.histogram(actual, bins=edges)[0] / len(actual)
    eps = 1e-9
    p = np.clip(p, eps, None)
    q = np.clip(q, eps, None)
    p /= p.sum()
    q /= q.sum()
    m = 0.5 * (p + q)
    return float(0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m)))


def analyze_feature_drift(
    X: np.ndarray,
    y: np.ndarray,
    features: Sequence[str],
    psi_warn: float = 0.10,
    psi_crit: float = 0.25,
) -> Dict[str, Any]:
    """Compare early vs. recent chronological halves for covariate & prior shift."""
    n = len(X)
    if n < 20:
        return {"status": "insufficient_data", "n": n, "features": {}}
    mid = n // 2
    X_early, X_recent = X[:mid], X[mid:]
    y_early, y_recent = y[:mid], y[mid:]
    drift_report: Dict[str, Any] = {}
    warnings, criticals = 0, 0

    for i, name in enumerate(features):
        psi = compute_psi(X_early[:, i], X_recent[:, i])
        ks_stat, ks_p = ks_2samp(X_early[:, i], X_recent[:, i])
        w_dist = float(wasserstein_distance(X_early[:, i], X_recent[:, i]))
        js_div = compute_js_divergence(X_early[:, i], X_recent[:, i])
        status = "ok"
        if psi >= psi_crit:
            status = "critical"
            criticals += 1
        elif psi >= psi_warn:
            status = "warning"
            warnings += 1
        drift_report[name] = {
            "psi": round(psi, 4),
            "ksStat": round(float(ks_stat), 4),
            "ksPValue": round(float(ks_p), 4),
            "wasserstein": round(w_dist, 4),
            "jsDivergence": round(js_div, 4),
            "status": status,
        }

    return {
        "status": "critical" if criticals > 0 else "warning" if warnings > 0 else "ok",
        "warningCount": warnings,
        "criticalCount": criticals,
        "priorShift": {
            "earlyPositiveRate": round(float(np.mean(y_early)), 4),
            "recentPositiveRate": round(float(np.mean(y_recent)), 4),
            "delta": round(float(np.mean(y_recent) - np.mean(y_early)), 4),
        },
        "features": drift_report,
    }


class AppClient:
    """Resilient HTTP client for PrideConnect ML endpoints."""

    def __init__(self, config: PipelineConfig) -> None:
        self.config = config
        self.session = requests.Session()
        retry = Retry(
            total=config.http_max_retries,
            backoff_factor=1.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["GET"]),
        )
        self.session.mount("https://", HTTPAdapter(max_retries=retry))
        self.session.headers.update({"Authorization": f"Bearer {config.sync_token}"})

    @timed("fetch_training_data")
    def fetch_training_data(self) -> Dict[str, Any]:
        url = f"{self.config.app_url}/api/ml/training-data"
        resp = self.session.get(
            url, timeout=(self.config.http_timeout_connect, self.config.http_timeout_read)
        )
        resp.raise_for_status()
        data = resp.json()
        if data.get("dataSource") != "LIVE_DATABASE":
            raise RuntimeError("Training data was not confirmed as a live database export.")
        features = list(data.get("featureNames") or [])
        if not features:
            raise RuntimeError("The export did not include any feature names.")
        if len(set(features)) != len(features):
            raise RuntimeError("The export contains duplicate feature names.")
        return data

    @timed("publish_artifacts")
    def publish_artifacts(self, artifacts: List[Dict[str, Any]], dry_run: bool = False) -> Any:
        url = f"{self.config.app_url}/api/ml/artifacts"
        payload = {
            "artifacts": artifacts,
            "dryRun": dry_run,
            "selectionMode": "CANDIDATE_REGISTRY" if dry_run else "PRODUCTION_SYNC",
        }
        resp = self.session.post(
            url,
            json=payload,
            timeout=(self.config.http_timeout_connect, self.config.http_timeout_read),
        )
        if not resp.ok:
            raise RuntimeError(f"Publish failed ({resp.status_code}): {resp.text[:500]}")
        try:
            return resp.json()
        except ValueError:
            return resp.text[:500]


# =============================================================================
# 3. METRICS, CALIBRATION & STATISTICAL EVALUATION
# =============================================================================


def calibration_diagnostics(
    y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10
) -> Dict[str, Any]:
    """Compute ECE, MCE, and reliability diagram bin statistics."""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece, mce = 0.0, 0.0
    n = len(y_true)
    if n == 0:
        return {"ece": 0.0, "mce": 0.0, "bins": []}

    bin_details: List[Dict[str, Any]] = []
    for i in range(n_bins):
        mask = (
            (y_prob >= bins[i]) & (y_prob < bins[i + 1])
            if i < n_bins - 1
            else (y_prob >= bins[i]) & (y_prob <= bins[i + 1])
        )
        count = int(np.sum(mask))
        if count > 0:
            bin_acc = float(np.mean(y_true[mask]))
            bin_conf = float(np.mean(y_prob[mask]))
            gap = abs(bin_acc - bin_conf)
            ece += (count / n) * gap
            mce = max(mce, gap)
            bin_details.append(
                {
                    "binLower": round(float(bins[i]), 2),
                    "binUpper": round(float(bins[i + 1]), 2),
                    "count": count,
                    "meanConfidence": round(bin_conf, 4),
                    "empiricalAccuracy": round(bin_acc, 4),
                }
            )
    return {"ece": float(ece), "mce": float(mce), "bins": bin_details}


def evaluate_posthoc_calibrators(
    y_true: np.ndarray, y_prob: np.ndarray
) -> Dict[str, Any]:
    """Compare uncalibrated OOF probabilities against Platt and Isotonic scaling."""
    if len(y_true) < 30 or len(set(y_true)) < 2:
        return {"status": "insufficient_data"}
    mid = len(y_true) // 2
    if len(set(y_true[:mid])) < 2 or len(set(y_true[mid:])) < 2:
        return {"status": "single_class_split"}

    p_train, y_train = y_prob[:mid], y_true[:mid]
    p_test, y_test = y_prob[mid:], y_true[mid:]

    # Platt scaling
    platt = LogisticRegression(C=1.0, solver="lbfgs")
    platt.fit(p_train.reshape(-1, 1), y_train)
    p_platt = platt.predict_proba(p_test.reshape(-1, 1))[:, 1]

    # Isotonic regression
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_train, y_train)
    p_iso = iso.predict(p_test)

    return {
        "rawBrier": round(float(brier_score_loss(y_test, p_test)), 4),
        "plattBrier": round(float(brier_score_loss(y_test, p_platt)), 4),
        "isotonicBrier": round(float(brier_score_loss(y_test, p_iso)), 4),
        "rawLogLoss": round(float(log_loss(y_test, p_test, labels=[0, 1])), 4),
        "plattLogLoss": round(float(log_loss(y_test, p_platt, labels=[0, 1])), 4),
    }


def find_optimal_thresholds(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    """Find optimal probability thresholds for F1, F2, and Youden's J."""
    if len(set(y_true)) < 2:
        return {"f1Threshold": 0.5, "f1Max": 0.0, "f2Threshold": 0.5, "youdenThreshold": 0.5}

    best_f1_t, best_f1 = 0.5, -1.0
    best_f2_t, best_f2 = 0.5, -1.0
    for thresh in np.linspace(0.1, 0.9, 81):
        preds = y_prob >= thresh
        f1 = float(f1_score(y_true, preds, zero_division=0))
        f2 = float(fbeta_score(y_true, preds, beta=2.0, zero_division=0))
        if f1 > best_f1:
            best_f1, best_f1_t = f1, float(thresh)
        if f2 > best_f2:
            best_f2, best_f2_t = f2, float(thresh)

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j_idx = int(np.argmax(tpr - fpr))
    youden_t = float(thresholds[j_idx]) if math.isfinite(thresholds[j_idx]) else 0.5

    return {
        "f1Threshold": round(best_f1_t, 3),
        "f1Max": round(best_f1, 4),
        "f2Threshold": round(best_f2_t, 3),
        "f2Max": round(best_f2, 4),
        "youdenThreshold": round(min(0.99, max(0.01, youden_t)), 3),
    }


def compute_classification_metrics(
    y_true: np.ndarray, y_prob: np.ndarray, ece_bins: int = 10
) -> Dict[str, Any]:
    """Compute comprehensive binary classification, calibration, and separation metrics."""
    has_both = len(set(y_true)) == 2
    preds = y_prob >= 0.5
    brier = float(brier_score_loss(y_true, y_prob))
    baseline_brier = float(brier_score_loss(y_true, np.full(len(y_true), np.mean(y_true))))
    bss = 1.0 - (brier / baseline_brier) if baseline_brier > 1e-12 else 0.0
    cal = calibration_diagnostics(y_true, y_prob, n_bins=ece_bins)
    thresholds = find_optimal_thresholds(y_true, y_prob)

    roc_auc = float(roc_auc_score(y_true, y_prob)) if has_both else None
    gini = (2.0 * roc_auc - 1.0) if roc_auc is not None else None
    ks_sep = (
        float(ks_2samp(y_prob[y_true == 1], y_prob[y_true == 0]).statistic) if has_both else None
    )

    tn, fp, fn, tp = (0, 0, 0, 0)
    if has_both:
        tn, fp, fn, tp = [int(x) for x in confusion_matrix(y_true, preds, labels=[0, 1]).ravel()]
    specificity = tn / max(1, tn + fp)
    npv = tn / max(1, tn + fn)

    return {
        "holdoutCount": int(len(y_true)),
        "positiveRate": float(np.mean(y_true)),
        "rocAuc": roc_auc,
        "gini": gini,
        "ksSeparation": ks_sep,
        "prAuc": float(average_precision_score(y_true, y_prob)) if has_both else None,
        "logLoss": float(log_loss(y_true, y_prob, labels=[0, 1])),
        "brierScore": brier,
        "baselineBrier": baseline_brier,
        "brierSkillScore": float(bss),
        "ece": cal["ece"],
        "mce": cal["mce"],
        "accuracy": float(accuracy_score(y_true, preds)),
        "balancedAccuracy": float(balanced_accuracy_score(y_true, preds)) if has_both else None,
        "precision": float(precision_score(y_true, preds, zero_division=0)),
        "recall": float(recall_score(y_true, preds, zero_division=0)),
        "specificity": float(specificity),
        "npv": float(npv),
        "f1": float(f1_score(y_true, preds, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, preds)) if has_both else 0.0,
        "cohenKappa": float(cohen_kappa_score(y_true, preds)) if has_both else 0.0,
        "confusionMatrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp},
        **thresholds,
    }


def bootstrap_metric_ci(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    metric_fn: Callable[[np.ndarray, np.ndarray], float],
    rounds: int = 500,
    seed: int = 17,
) -> Optional[List[float]]:
    """Compute 95% bootstrap confidence interval for a given metric."""
    if len(set(y_true)) < 2:
        return None
    rng = np.random.default_rng(seed)
    scores: List[float] = []
    n = len(y_true)
    for _ in range(rounds):
        idx = rng.integers(0, n, n)
        if len(set(y_true[idx])) == 2:
            scores.append(float(metric_fn(y_true[idx], y_prob[idx])))
    if len(scores) < 30:
        return None
    return [float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))]


def compute_ranking_metrics(
    rows: Sequence[Dict[str, Any]], probs: np.ndarray, k: int = 5
) -> Dict[str, Optional[float]]:
    """Compute per-sender HitRate@K, NDCG@K, MRR, and MAP@K."""
    by_sender: Dict[str, List[Tuple[float, int]]] = {}
    for row, prob in zip(rows, probs):
        sender = str(row.get("senderKey", ""))
        if sender:
            by_sender.setdefault(sender, []).append((float(prob), int(row["label"])))

    hits, ndcgs, mrrs, maps = [], [], [], []
    for items in by_sender.values():
        if len(items) < 2 or sum(lbl for _, lbl in items) == 0:
            continue
        items_sorted = sorted(items, key=lambda x: x[0], reverse=True)[:k]
        hits.append(1.0 if any(lbl == 1 for _, lbl in items_sorted) else 0.0)

        dcg = sum(lbl / math.log2(idx + 2) for idx, (_, lbl) in enumerate(items_sorted))
        ideal_sorted = sorted([lbl for _, lbl in items], reverse=True)[:k]
        idcg = sum(lbl / math.log2(idx + 2) for idx, lbl in enumerate(ideal_sorted))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

        rr = 0.0
        for idx, (_, lbl) in enumerate(items_sorted):
            if lbl == 1:
                rr = 1.0 / (idx + 1)
                break
        mrrs.append(rr)

        rel_count, ap_sum = 0, 0.0
        for idx, (_, lbl) in enumerate(items_sorted):
            if lbl == 1:
                rel_count += 1
                ap_sum += rel_count / (idx + 1)
        maps.append(ap_sum / max(1, sum(lbl for _, lbl in items_sorted)))

    return {
        f"hitRate@{k}": float(np.mean(hits)) if hits else None,
        f"ndcg@{k}": float(np.mean(ndcgs)) if ndcgs else None,
        "mrr": float(np.mean(mrrs)) if mrrs else None,
        f"map@{k}": float(np.mean(maps)) if maps else None,
        "evaluatedSenders": len(hits),
    }


# =============================================================================
# 4. LINEAR & TABULAR MODELS
# =============================================================================


def temporal_folds(n: int) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Generate expanding-window chronological cross-validation folds."""
    k = 4 if n >= 80 else 3 if n >= 40 else 2 if n >= 24 else 1
    start = int(n * (0.8 if k == 1 else 0.55))
    size = max(1, (n - start) // k)
    folds: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(k):
        train_end = start + i * size
        test_end = n if i == k - 1 else train_end + size
        if train_end >= 1 and test_end > train_end:
            folds.append((np.arange(train_end), np.arange(train_end, test_end)))
    return folds


class LinearModel:
    """Linear logistic model with scaling folded into raw-feature weights."""

    def __init__(self, coef: np.ndarray, bias: float, scale: np.ndarray) -> None:
        self.coef = coef
        self.bias = bias
        self.scale = scale

    def predict(self, X: np.ndarray) -> np.ndarray:
        return expit(X @ self.coef + self.bias)


def linear_fitter(
    C: float, penalty: str = "l2", l1_ratio: Optional[float] = None, seed: int = 17
) -> Callable:
    """Return a fitter function that standardizes features and folds scaling back."""

    def fit(
        X: np.ndarray, y: np.ndarray, sample_weight: Optional[np.ndarray] = None
    ) -> LinearModel:
        mean = X.mean(axis=0)
        scale = np.where(X.std(axis=0) < 1e-8, 1.0, X.std(axis=0))
        solver = "saga" if penalty in ("l1", "elasticnet") else "lbfgs"
        kwargs: Dict[str, Any] = {
            "class_weight": "balanced",
            "C": C,
            "penalty": penalty,
            "solver": solver,
            "max_iter": 3000,
            "random_state": seed,
        }
        if penalty == "elasticnet":
            kwargs["l1_ratio"] = l1_ratio if l1_ratio is not None else 0.5

        model = LogisticRegression(**kwargs)
        model.fit((X - mean) / scale, y, sample_weight=sample_weight)
        raw_coef = model.coef_[0] / scale
        raw_bias = float(model.intercept_[0] - np.sum(model.coef_[0] * mean / scale))
        return LinearModel(raw_coef, raw_bias, scale)

    return fit


class BenchmarkModel:
    """Wrapper for non-linear benchmark models."""

    def __init__(self, estimator: Any, X: np.ndarray, y: np.ndarray) -> None:
        self.model = estimator
        self.model.fit(X, y)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.model.predict_proba(X)[:, 1]


def out_of_fold(
    X: np.ndarray,
    y: np.ndarray,
    fit_fn: Callable,
    sample_weight: Optional[np.ndarray] = None,
) -> Optional[Tuple[np.ndarray, np.ndarray, List[Dict[str, Any]]]]:
    """Run expanding-window temporal CV and pool out-of-fold predictions."""
    truths, scores, per_fold = [], [], []
    for fold_idx, (train_idx, test_idx) in enumerate(temporal_folds(len(y))):
        if len(set(y[train_idx])) < 2:
            continue
        w_train = sample_weight[train_idx] if sample_weight is not None else None
        try:
            model = fit_fn(X[train_idx], y[train_idx], w_train)
        except TypeError:
            model = fit_fn(X[train_idx], y[train_idx])
        probs = model.predict(X[test_idx])
        truths.append(y[test_idx])
        scores.append(probs)
        per_fold.append(
            {
                "fold": fold_idx + 1,
                "trainCount": int(len(train_idx)),
                "testCount": int(len(test_idx)),
                "rocAuc": float(roc_auc_score(y[test_idx], probs))
                if len(set(y[test_idx])) == 2
                else None,
            }
        )
    if not truths:
        return None
    return np.concatenate(truths), np.concatenate(scores), per_fold


def select_regularization(
    X: np.ndarray, y: np.ndarray, sample_weight: np.ndarray, config: PipelineConfig
) -> Tuple[float, str, Optional[float], List[Dict[str, Any]]]:
    """Grid search over C, penalty, and ElasticNet l1_ratio using pooled temporal CV."""
    search: List[Dict[str, Any]] = []
    for penalty in config.lr_penalties:
        ratios = config.lr_l1_ratios if penalty == "elasticnet" else (None,)
        for l1_ratio in ratios:
            for C in config.lr_c_grid:
                result = out_of_fold(
                    X,
                    y,
                    linear_fitter(C, penalty=penalty, l1_ratio=l1_ratio, seed=config.seed),
                    sample_weight=sample_weight,
                )
                if not result or len(set(result[0])) < 2:
                    continue
                truth, probs, _ = result
                search.append(
                    {
                        "C": C,
                        "penalty": penalty,
                        "l1Ratio": l1_ratio,
                        "rocAuc": float(roc_auc_score(truth, probs)),
                        "logLoss": float(log_loss(truth, probs, labels=[0, 1])),
                        "brierScore": float(brier_score_loss(truth, probs)),
                    }
                )
    if not search:
        return 1.0, "l2", None, []
    best = max(search, key=lambda item: (round(item["rocAuc"], 4), -item["logLoss"], -item["C"]))
    return best["C"], best["penalty"], best["l1Ratio"], search


@timed("train_linear_artifact")
def train_linear_artifact(
    rows: Optional[List[Dict[str, Any]]],
    kind: str,
    features: Sequence[str],
    data_source: str,
    minimum_rows: int,
    minimum_each_class: int,
    config: PipelineConfig,
    report: Dict[str, Any],
) -> Optional[Dict[str, Any]]:
    """Train, evaluate, benchmark, calibrate, and package a linear model artifact."""
    logger = logging.getLogger("prideconnect.ml")
    clean_rows = usable_rows(rows, needs_features=True)
    y = np.array([int(row["label"]) for row in clean_rows], dtype=int)
    positive, negative = int(y.sum()), int(len(y) - y.sum())

    if len(clean_rows) < minimum_rows or min(positive, negative) < minimum_each_class:
        logger.warning(
            "Skipping %s: need n>=%d and >=%d per class; got n=%d (pos=%d, neg=%d).",
            kind,
            minimum_rows,
            minimum_each_class,
            len(clean_rows),
            positive,
            negative,
        )
        return None

    X = design_matrix(clean_rows, features)
    sample_weights = compute_time_decay_weights(clean_rows, config.time_decay_half_life_days)
    drift = analyze_feature_drift(
        X, y, features, psi_warn=config.psi_warning_threshold, psi_crit=config.psi_critical_threshold
    )

    best_c, best_penalty, best_l1_ratio, search = select_regularization(X, y, sample_weights, config)
    result = out_of_fold(
        X,
        y,
        linear_fitter(best_c, penalty=best_penalty, l1_ratio=best_l1_ratio, seed=config.seed),
        sample_weight=sample_weights,
    )

    metrics: Dict[str, Any] = {
        "holdoutCount": 0,
        "rocAuc": None,
        "brierScore": None,
        "accuracy": None,
    }
    diagnostics: Dict[str, Any] = {
        "chosenC": best_c,
        "chosenPenalty": best_penalty,
        "chosenL1Ratio": best_l1_ratio,
        "searchCount": len(search),
        "folds": [],
        "rocAucCi95": None,
        "prAucCi95": None,
        "posthocCalibration": None,
        "benchmarks": {},
        "drift": drift,
    }

    if result:
        truth, probs, folds = result
        metrics = compute_classification_metrics(truth, probs, ece_bins=config.ece_bins)
        diagnostics["folds"] = folds
        diagnostics["rocAucCi95"] = bootstrap_metric_ci(
            truth, probs, roc_auc_score, rounds=config.bootstrap_rounds, seed=config.seed
        )
        diagnostics["prAucCi95"] = bootstrap_metric_ci(
            truth, probs, average_precision_score, rounds=config.bootstrap_rounds, seed=config.seed
        )
        diagnostics["posthocCalibration"] = evaluate_posthoc_calibrators(truth, probs)

        benchmarks = {
            "histGbdt": lambda a, b: BenchmarkModel(
                HistGradientBoostingClassifier(
                    max_depth=3,
                    learning_rate=0.05,
                    max_iter=150,
                    l2_regularization=1.0,
                    min_samples_leaf=max(3, min(20, len(b) // 10)),
                    class_weight="balanced",
                    random_state=config.seed,
                ),
                a,
                b,
            ),
            "randomForest": lambda a, b: BenchmarkModel(
                RandomForestClassifier(
                    n_estimators=100,
                    max_depth=4,
                    min_samples_leaf=max(2, len(b) // 15),
                    class_weight="balanced",
                    random_state=config.seed,
                ),
                a,
                b,
            ),
            "extraTrees": lambda a, b: BenchmarkModel(
                ExtraTreesClassifier(
                    n_estimators=100,
                    max_depth=4,
                    min_samples_leaf=max(2, len(b) // 15),
                    class_weight="balanced",
                    random_state=config.seed,
                ),
                a,
                b,
            ),
            "mlp": lambda a, b: BenchmarkModel(
                MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=config.seed),
                (a - a.mean(axis=0)) / np.where(a.std(axis=0) < 1e-8, 1.0, a.std(axis=0)),
                b,
            ),
            "priorBaseline": lambda a, b: BenchmarkModel(DummyClassifier(strategy="prior"), a, b),
        }
        for b_name, b_fitter in benchmarks.items():
            try:
                b_res = out_of_fold(X, y, b_fitter)
                if b_res and len(set(b_res[0])) == 2:
                    diagnostics["benchmarks"][b_name] = {
                        "rocAuc": float(roc_auc_score(b_res[0], b_res[1])),
                        "brierScore": float(brier_score_loss(b_res[0], b_res[1])),
                    }
            except Exception as exc:
                logger.debug("%s benchmark %s skipped: %s", kind, b_name, exc)

    passed_gate = metrics["rocAuc"] is None or metrics["rocAuc"] >= config.lr_min_auc
    if not passed_gate:
        logger.warning(
            "%s: OOF AUC %.3f < %.3f (marked inactive candidate).",
            kind,
            metrics["rocAuc"],
            config.lr_min_auc,
        )
        if not (config.dry_run and config.publish_after_dry_run):
            report["linear"][kind] = {"metrics": metrics, **diagnostics, "published": False}
            return None

    final = linear_fitter(
        best_c, penalty=best_penalty, l1_ratio=best_l1_ratio, seed=config.seed
    )(X, y, sample_weight=sample_weights)
    standardized = final.coef * final.scale
    order = np.argsort(-np.abs(standardized))[:8]
    diagnostics["topFeatures"] = [
        {"feature": features[i], "standardizedWeight": float(standardized[i])} for i in order
    ]

    try:
        mean = X.mean(axis=0)
        scale = np.where(X.std(axis=0) < 1e-8, 1.0, X.std(axis=0))
        X_std = (X - mean) / scale
        solver = "saga" if best_penalty in ("l1", "elasticnet") else "lbfgs"
        perm_kwargs: Dict[str, Any] = {
            "class_weight": "balanced",
            "C": best_c,
            "penalty": best_penalty,
            "solver": solver,
            "max_iter": 2000,
            "random_state": config.seed,
        }
        if best_penalty == "elasticnet":
            perm_kwargs["l1_ratio"] = best_l1_ratio
        perm_model = LogisticRegression(**perm_kwargs)
        perm_model.fit(X_std, y)
        perm = permutation_importance(
            perm_model, X_std, y, scoring="roc_auc", n_repeats=10, random_state=config.seed
        )
        perm_order = np.argsort(-perm.importances_mean)[:6]
        diagnostics["permutationImportance"] = [
            {
                "feature": features[i],
                "meanAucDrop": float(perm.importances_mean[i]),
                "std": float(perm.importances_std[i]),
            }
            for i in perm_order
        ]
    except Exception as exc:
        logger.debug("%s permutation importance skipped: %s", kind, exc)

    report["linear"][kind] = {"metrics": metrics, **diagnostics, "published": True}

    minimum_confident = 12 if kind == "ACCEPTANCE" else 8
    confidence = float(
        min(1.0, max(0.0, (len(clean_rows) - minimum_confident) / (50 - minimum_confident)))
    )

    is_active = bool(passed_gate and not config.dry_run)
    return {
        "type": kind,
        "dataSource": data_source,
        "active": is_active,
        "deploymentStatus": "CANDIDATE" if config.dry_run else ("PRODUCTION" if passed_gate else "SHADOW"),
        "version": config.run_id,
        "featureNames": list(features),
        "weights": {name: float(w) for name, w in zip(features, final.coef)},
        "bias": float(final.bias),
        "sampleCount": len(clean_rows),
        "positiveCount": positive,
        "negativeCount": negative,
        "confidence": confidence if passed_gate else 0.0,
        "metrics": metrics,
    }


# =============================================================================
# 5. NEURAL COLLABORATIVE FILTERING (NCF)
# =============================================================================


class NCF(nn.Module):
    """Neural Collaborative Filtering model matching PrideConnect's inference schema."""

    def __init__(
        self, num_users: int, embedding_dim: int, hidden_dim: int, num_avail: int, dropout: float = 0.0
    ) -> None:
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        self.dropout = nn.Dropout(dropout)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2 + 2 + num_avail, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(
        self,
        sender_idx: torch.Tensor,
        recipient_idx: torch.Tensor,
        request_type_onehot: torch.Tensor,
        availability: torch.Tensor,
    ) -> torch.Tensor:
        joined = torch.cat(
            [
                self.user_embedding(sender_idx),
                self.user_embedding(recipient_idx),
                request_type_onehot,
                availability,
            ],
            dim=1,
        )
        return self.mlp(self.dropout(joined)).squeeze(1)


def member_index(rows: Sequence[Dict[str, Any]]) -> Dict[str, int]:
    keys = sorted({str(r["senderKey"]) for r in rows} | {str(r["recipientKey"]) for r in rows})
    return {key: i for i, key in enumerate(keys)}


def evaluable_rows(rows: Sequence[Dict[str, Any]], indices: Dict[str, int]) -> List[Dict[str, Any]]:
    return [r for r in rows if str(r["senderKey"]) in indices and str(r["recipientKey"]) in indices]


def to_tensors(
    rows: Sequence[Dict[str, Any]], indices: Dict[str, int], avail_features: Sequence[str]
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    sender = torch.tensor([indices[str(r["senderKey"])] for r in rows], dtype=torch.long)
    recipient = torch.tensor([indices[str(r["recipientKey"])] for r in rows], dtype=torch.long)
    kind = torch.tensor(
        [[1.0, 0.0] if r.get("isMentorship") else [0.0, 1.0] for r in rows], dtype=torch.float32
    )
    availability = torch.tensor(
        [[to_float((r.get("features") or {}).get(name)) for name in avail_features] for r in rows],
        dtype=torch.float32,
    )
    labels = torch.tensor([float(r["label"]) for r in rows], dtype=torch.float32)
    return sender, recipient, kind, availability, labels


def predict_ncf_probs(
    model: NCF, rows: Sequence[Dict[str, Any]], indices: Dict[str, int], avail_features: Sequence[str]
) -> Tuple[np.ndarray, np.ndarray]:
    sender, recipient, kind, availability, labels = to_tensors(rows, indices, avail_features)
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(sender, recipient, kind, availability)).numpy()
    return labels.numpy().astype(int), probs


def fit_ncf(
    train_rows: Sequence[Dict[str, Any]],
    indices: Dict[str, int],
    lr: float,
    weight_decay: float,
    dropout: float,
    epochs: int,
    config: PipelineConfig,
    valid_rows: Optional[Sequence[Dict[str, Any]]] = None,
) -> Tuple[NCF, int, float]:
    seed_everything(config.seed)
    model = NCF(
        num_users=len(indices),
        embedding_dim=config.ncf_embedding_dim,
        hidden_dim=config.ncf_hidden_dim,
        num_avail=len(config.availability_features),
        dropout=dropout,
    )
    sender, recipient, kind, availability, labels = to_tensors(
        train_rows, indices, config.availability_features
    )
    eps = config.ncf_label_smoothing
    smoothed_labels = labels * (1.0 - eps) + 0.5 * eps
    positive = max(1.0, float(labels.sum()))
    negative = max(1.0, float(len(labels) - labels.sum()))
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(negative / positive))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    generator = torch.Generator().manual_seed(config.seed)

    best_auc, best_epoch, best_state, stale = -1.0, epochs, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(labels), generator=generator)
        for start in range(0, len(order), config.ncf_batch_size):
            batch = order[start : start + config.ncf_batch_size]
            optimizer.zero_grad()
            loss = criterion(
                model(sender[batch], recipient[batch], kind[batch], availability[batch]),
                smoothed_labels[batch],
            )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), config.ncf_grad_clip)
            optimizer.step()
        scheduler.step()

        if valid_rows is not None:
            truth, probs = predict_ncf_probs(model, valid_rows, indices, config.availability_features)
            auc = float(roc_auc_score(truth, probs))
            if auc > best_auc + 1e-4:
                best_auc, best_epoch, stale = auc, epoch, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                stale += 1
                if stale >= config.ncf_patience:
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_epoch, best_auc


def select_ncf_config(
    train_rows: Sequence[Dict[str, Any]], config: PipelineConfig
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    logger = logging.getLogger("prideconnect.ml")
    inner = int(len(train_rows) * 0.85)
    fit_rows, valid_rows = train_rows[:inner], train_rows[inner:]
    indices = member_index(fit_rows)
    valid = evaluable_rows(valid_rows, indices)
    default_cfg = {"lr": 0.005, "weightDecay": 1e-2, "dropout": 0.0, "epochs": 120}

    if len(valid) < 15 or len({int(r["label"]) for r in valid}) < 2:
        logger.info("NCF: validation slice too small for tuning; using default hyperparameters.")
        return default_cfg, []

    trials: List[Dict[str, Any]] = []
    for lr in config.ncf_lr_grid:
        for wd in config.ncf_weight_decays:
            for dropout in config.ncf_dropouts:
                _, epoch, auc = fit_ncf(
                    fit_rows, indices, lr, wd, dropout, config.ncf_max_epochs, config, valid
                )
                trials.append(
                    {
                        "lr": lr,
                        "weightDecay": wd,
                        "dropout": dropout,
                        "epochs": epoch,
                        "validationAuc": auc,
                    }
                )
    best = max(trials, key=lambda t: t["validationAuc"])
    logger.info(
        "NCF tuning best: lr=%s wd=%s dropout=%s epochs=%d val_auc=%.3f",
        best["lr"],
        best["weightDecay"],
        best["dropout"],
        best["epochs"],
        best["validationAuc"],
    )
    return {
        "lr": best["lr"],
        "weightDecay": best["weightDecay"],
        "dropout": best["dropout"],
        "epochs": max(20, int(round(best["epochs"] * 1.15))),
    }, trials


@timed("train_ncf_artifact")
def train_ncf_artifact(
    raw_ncf_rows: Optional[List[Dict[str, Any]]],
    data_source: str,
    config: PipelineConfig,
    report: Dict[str, Any],
) -> Optional[Dict[str, Any]]:
    logger = logging.getLogger("prideconnect.ml")
    ncf_rows = usable_rows(raw_ncf_rows, needs_features=False)
    ncf_rows = [r for r in ncf_rows if r.get("senderKey") and r.get("recipientKey")][
        -config.ncf_max_rows :
    ]
    member_total = len({str(r["senderKey"]) for r in ncf_rows} | {str(r["recipientKey"]) for r in ncf_rows})

    if len(ncf_rows) < config.ncf_min_rows or member_total < config.ncf_min_members:
        logger.warning(
            "NCF skipped: need >=%d rows and >=%d members; got %d rows and %d members.",
            config.ncf_min_rows,
            config.ncf_min_members,
            len(ncf_rows),
            member_total,
        )
        return None

    cut = int(len(ncf_rows) * 0.8)
    train_rows, holdout_rows = ncf_rows[:cut], ncf_rows[cut:]
    train_index = member_index(train_rows)
    evaluable = evaluable_rows(holdout_rows, train_index)
    holdout_coverage = len(evaluable) / max(1, len(holdout_rows))

    if (
        len(evaluable) < config.ncf_min_holdout
        or len({int(r["label"]) for r in evaluable}) < 2
        or holdout_coverage < config.ncf_min_coverage
    ):
        logger.warning(
            "NCF ineligible for evaluation: evaluable=%d coverage=%.1f%%.",
            len(evaluable),
            holdout_coverage * 100,
        )
        return None

    cfg, trials = select_ncf_config(train_rows, config)
    check_model, _, _ = fit_ncf(
        train_rows, train_index, cfg["lr"], cfg["weightDecay"], cfg["dropout"], cfg["epochs"], config
    )
    hold_true, hold_probs = predict_ncf_probs(
        check_model, evaluable, train_index, config.availability_features
    )
    hold_metrics = compute_classification_metrics(hold_true, hold_probs, ece_bins=config.ece_bins)
    ranking_metrics = compute_ranking_metrics(evaluable, hold_probs, k=5)

    hold_auc = hold_metrics["rocAuc"] or 0.0
    ncf_passed = (
        hold_auc >= config.ncf_min_auc
        and holdout_coverage >= config.ncf_min_coverage
        and len(evaluable) >= config.ncf_min_holdout
    )

    all_index = member_index(ncf_rows)
    final_model, _, _ = fit_ncf(
        ncf_rows, all_index, cfg["lr"], cfg["weightDecay"], cfg["dropout"], cfg["epochs"], config
    )
    state = final_model.state_dict()
    emb_tensor = state["user_embedding.weight"]
    emb_norms = torch.norm(emb_tensor, dim=1)
    user_embeddings = {key: [float(v) for v in emb_tensor[idx].tolist()] for key, idx in all_index.items()}

    ncf_confidence = (
        float(min(1.0, len(ncf_rows) / 500.0) * min(1.0, max(0.0, (hold_auc - 0.5) / 0.15)))
        if ncf_passed
        else 0.0
    )

    report["ncf"] = {
        "config": cfg,
        "trials": trials,
        "members": len(all_index),
        "active": ncf_passed and not config.dry_run,
        "rankingMetrics": ranking_metrics,
        "embeddingNormMean": float(emb_norms.mean().item()),
        "embeddingNormStd": float(emb_norms.std().item()),
    }

    return {
        "type": "NCF",
        "dataSource": data_source,
        "active": bool(ncf_passed and not config.dry_run),
        "deploymentStatus": "CANDIDATE" if config.dry_run else ("PRODUCTION" if ncf_passed else "SHADOW"),
        "version": config.run_id,
        "featureNames": ["senderKey", "recipientKey", "requestTypeOneHot", *config.availability_features],
        "weights": {
            "userEmbeddings": user_embeddings,
            "layer1": {
                "weights": state["mlp.0.weight"].tolist(),
                "bias": state["mlp.0.bias"].tolist(),
            },
            "layer2": {
                "weights": state["mlp.2.weight"].tolist(),
                "bias": state["mlp.2.bias"].tolist(),
            },
        },
        "bias": 0.0,
        "sampleCount": len(ncf_rows),
        "positiveCount": int(sum(int(r["label"]) == 1 for r in ncf_rows)),
        "negativeCount": int(sum(int(r["label"]) == 0 for r in ncf_rows)),
        "confidence": ncf_confidence,
        "metrics": {
            "holdoutRocAuc": hold_metrics["rocAuc"],
            "holdoutPrAuc": hold_metrics["prAuc"],
            "holdoutBrierScore": hold_metrics["brierScore"],
            "holdoutAccuracy": hold_metrics["accuracy"],
            "holdoutEce": hold_metrics["ece"],
            "holdoutCoverage": float(holdout_coverage),
            "holdoutCount": len(evaluable),
            **ranking_metrics,
        },
    }


# =============================================================================
# 6. ARTIFACT VALIDATION & CHAMPION-CHALLENGER MONITORING
# =============================================================================


def assert_finite(node: Any, path: str = "artifact") -> None:
    """Recursively verify that all numeric values in an artifact are finite."""
    if isinstance(node, dict):
        for key, value in node.items():
            assert_finite(value, f"{path}.{key}")
    elif isinstance(node, (list, tuple)):
        for i, value in enumerate(node):
            assert_finite(value, f"{path}[{i}]")
    elif isinstance(node, float) and not math.isfinite(node):
        raise ValueError(f"Non-finite value at {path}")


def compare_with_previous_report(
    storage_dir: Path, current_report: Dict[str, Any], tolerance: float = 0.02
) -> Dict[str, Any]:
    """Compare current run metrics against the most recent saved report."""
    reports = sorted(storage_dir.glob("report-*.json"))
    if not reports:
        return {"status": "no_previous_report"}
    try:
        prev = json.loads(reports[-1].read_text(encoding="utf-8"))
    except Exception as exc:
        return {"status": "previous_report_unreadable", "error": str(exc)}

    deltas: Dict[str, Any] = {}
    for kind in ("ACCEPTANCE", "RELEVANCE"):
        curr_auc = (current_report.get("linear", {}).get(kind, {}).get("metrics") or {}).get("rocAuc")
        prev_auc = (prev.get("linear", {}).get(kind, {}).get("metrics") or {}).get("rocAuc")
        if curr_auc is not None and prev_auc is not None:
            diff = curr_auc - prev_auc
            deltas[kind] = {
                "previousAuc": prev_auc,
                "currentAuc": curr_auc,
                "delta": round(diff, 4),
                "regressionAlert": diff < -tolerance,
            }
    return {"status": "compared", "previousRunId": prev.get("runId"), "deltas": deltas}


# =============================================================================
# 7. AUTOMATED TEST SUITE
# =============================================================================


class TestPrideConnectPipeline(unittest.TestCase):
    """Self-contained unit tests for data, math, models, and artifact integrity."""

    def test_usable_rows_filtering_and_sorting(self) -> None:
        raw = [
            {"label": 1, "timestamp": "2026-02-02", "features": {"a": 1}},
            {"label": 2, "timestamp": "2026-02-01", "features": {"a": 1}},
            {"label": 0, "timestamp": "2026-02-01", "features": None},
            {"label": 0, "timestamp": "2026-01-01", "features": {"a": 0}},
        ]
        kept = usable_rows(raw, needs_features=True)
        self.assertEqual(len(kept), 2)
        self.assertEqual(kept[0]["timestamp"], "2026-01-01")

    def test_time_decay_weights_normalization(self) -> None:
        rows = [
            {"timestamp": "2026-01-01T00:00:00Z"},
            {"timestamp": "2026-04-01T00:00:00Z"},
        ]
        w = compute_time_decay_weights(rows, half_life_days=90.0)
        self.assertAlmostEqual(float(np.mean(w)), 1.0, places=6)
        self.assertGreater(w[1], w[0])

    def test_linear_weight_folding_equivalence(self) -> None:
        rng = np.random.default_rng(42)
        X = rng.normal(loc=10.0, scale=5.0, size=(60, 3))
        y = (X[:, 0] - 0.5 * X[:, 1] > 10.0).astype(int)
        model = linear_fitter(C=1.0, penalty="elasticnet", l1_ratio=0.5, seed=42)(X, y)
        probs = model.predict(X)
        self.assertEqual(probs.shape, (60,))
        self.assertTrue(np.all((probs > 0.0) & (probs < 1.0)))

    def test_temporal_folds_no_leakage(self) -> None:
        folds = temporal_folds(100)
        self.assertGreaterEqual(len(folds), 2)
        for train_idx, test_idx in folds:
            self.assertLess(np.max(train_idx), np.min(test_idx))

    def test_ncf_forward_and_export_shapes(self) -> None:
        model = NCF(num_users=5, embedding_dim=8, hidden_dim=16, num_avail=4, dropout=0.0)
        s = torch.tensor([0, 1])
        r = torch.tensor([2, 3])
        k = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
        a = torch.zeros((2, 4))
        out = model(s, r, k, a)
        self.assertEqual(out.shape, (2,))
        state = model.state_dict()
        self.assertEqual(state["mlp.0.weight"].shape, (16, 22))
        self.assertEqual(state["mlp.2.weight"].shape, (1, 16))

    def test_assert_finite_catches_nan(self) -> None:
        with self.assertRaises(ValueError):
            assert_finite({"a": [1.0, float("nan")]})


def run_self_tests() -> None:
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestPrideConnectPipeline)
    result = unittest.TextTestRunner(verbosity=1).run(suite)
    if not result.wasSuccessful():
        raise RuntimeError("Automated pipeline self-tests failed.")


# =============================================================================
# 8. ORCHESTRATION ENTRYPOINT
# =============================================================================


def resolve_storage_dir(config: PipelineConfig) -> Path:
    logger = logging.getLogger("prideconnect.ml")
    if IN_COLAB:
        try:
            drive.mount("/content/drive")
            config.drive_dir.mkdir(parents=True, exist_ok=True)
            return config.drive_dir
        except Exception as exc:
            logger.warning("Drive unavailable, falling back to local storage: %s", exc)
    config.fallback_dir.mkdir(parents=True, exist_ok=True)
    return config.fallback_dir


def main(argv: Optional[Sequence[str]] = None) -> None:
    parser = argparse.ArgumentParser(description="PrideConnect ML Training & Publishing Pipeline")
    parser.add_argument(
        "--dry-run",
        action="store_true",
        help="Run training and publish as inactive CANDIDATE artifacts so the app can choose.",
    )
    parser.add_argument(
        "--no-publish",
        action="store_true",
        help="Do not publish to the app at all (local save only).",
    )
    parser.add_argument("--skip-tests", action="store_true", help="Skip automated unit tests.")
    args, _ = parser.parse_known_args(argv)

    config = PipelineConfig(
        dry_run=args.dry_run,
        publish=not args.no_publish,
        publish_after_dry_run=not args.no_publish,
        run_tests=not args.skip_tests,
    )
    seed_everything(config.seed)
    storage_dir = resolve_storage_dir(config)
    logger = setup_logging(log_dir=storage_dir)

    if config.run_tests:
        logger.info("Running automated self-tests...")
        run_self_tests()

    config.app_url = read_secret(
        "PRIDECONNECT_APP_URL",
        "Preview app base URL (example: https://your-preview.vercel.app): ",
    ).rstrip("/")
    config.sync_token = read_secret(
        "ML_COLAB_SYNC_TOKEN", "ML_COLAB_SYNC_TOKEN (input hidden): ", hidden=True
    )
    config.validate()

    client = AppClient(config)
    data = client.fetch_training_data()
    features = list(data["featureNames"])
    data_source = data["dataSource"]

    logger.info(
        "Loaded data: features=%d acceptance=%d relevance=%d ncf=%d",
        len(features),
        len(data.get("acceptance") or []),
        len(data.get("relevance") or []),
        len(data.get("ncf") or []),
    )

    report: Dict[str, Any] = {
        "runId": config.run_id,
        "config": asdict(config),
        "linear": {},
        "ncf": {},
    }
    report["config"]["drive_dir"] = str(config.drive_dir)
    report["config"]["fallback_dir"] = str(config.fallback_dir)
    report["config"]["sync_token"] = "***REDACTED***"

    artifacts: List[Dict[str, Any]] = []
    for rows, kind, min_rows, min_class in [
        (data.get("acceptance"), "ACCEPTANCE", 12, 4),
        (data.get("relevance"), "RELEVANCE", 8, 3),
    ]:
        art = train_linear_artifact(
            rows, kind, features, data_source, min_rows, min_class, config, report
        )
        if art:
            artifacts.append(art)

    ncf_art = train_ncf_artifact(data.get("ncf"), data_source, config, report)
    if ncf_art:
        artifacts.append(ncf_art)

    if not artifacts:
        raise RuntimeError(
            "No model met its minimum evidence and quality checks. Collect more outcomes/feedback."
        )

    for artifact in artifacts:
        assert_finite(artifact, artifact["type"])

    report["championChallenger"] = compare_with_previous_report(
        storage_dir, report, tolerance=config.non_regression_tolerance
    )

    artifact_path = storage_dir / f"models-{config.run_id}.json"
    artifact_path.write_text(
        json.dumps({"createdAt": config.run_id, "artifacts": artifacts}, indent=2),
        encoding="utf-8",
    )
    report_path = storage_dir / f"report-{config.run_id}.json"
    report_path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")

    logger.info("Saved model artifacts to %s", artifact_path)
    logger.info("Saved evaluation report to %s", report_path)

    if config.publish and (not config.dry_run or config.publish_after_dry_run):
        pub_result = client.publish_artifacts(artifacts, dry_run=config.dry_run)
        logger.info(
            "Published %s artifacts to app: %s",
            "CANDIDATE (dry-run)" if config.dry_run else "PRODUCTION",
            pub_result,
        )
    else:
        logger.info("Publishing disabled (--no-publish); artifacts saved locally only.")


if __name__ == "__main__":
    main()s